In [ ]:
from sympy import Array, tensorproduct
from sympy.abc import x,y,z,t,d
A = Array([x, y, z, t],(d,d))
B = Array([x, y, z, t],(d,d))
print(tensorproduct(A, B))

In [ ]:
from sympy import symbols, IndexedBase, Idx
dim1, dim2 = symbols('d_1, d_2', integer=True)
M = IndexedBase('M',shape=(dim1, dim2))
i, j = symbols('i j', cls=Idx)
x = IndexedBase('x', shape=(dim2,))
(M[i, j]*x[j] ).shape

In [ ]:
n, m = symbols('n m', integer=True)
i = Idx('i', m)
j = Idx('j', n)
M[i, j].shape

In [ ]:
M[i, j].ranges

In [ ]:
from sympy.tensor import get_indices, get_contraction_structure
get_indices(A[i, j, j])
# get_contraction_structure(A[i, j, j])

In [19]:
from sympy.tensor.indexed import Indexed

M = IndexedBase('M', shape=(dim1, dim2))
x = IndexedBase('x', shape=(dim2,))
i, j = symbols('i j', cls=Idx)
expr = M[i, j] * x[j]   # contraction over j

# Free/open indices for Indexed expressions: those appearing once
counts = {}
for atom in expr.atoms(Indexed):
    for idx in atom.indices:
        counts[idx] = counts.get(idx, 0) + 1

free = tuple(idx for idx, c in counts.items() if c == 1)
free  # -> (i,)

(i,)

In [ ]:
from sympy.tensor.indexed import Indexed
from sympy import summation


def HaarIntegral(expr, varName):
    # integrate expr where varName is the variable of integration
    # first give all open indices of the expression, and print out the indices connected to varName
    # next perform all contractions over dummy indices connected to varName

    # Collect index counts for all Indexed atoms
    counts = {}
    for atom in expr.atoms(Indexed):
        for idx in atom.indices:
            counts[idx] = counts.get(idx, 0) + 1

    open_indices = tuple(idx for idx, c in counts.items() if c == 1)

    # Indices directly attached to the chosen base (varName)
    connected = set()
    for atom in expr.atoms(Indexed):
        if str(atom.base) == str(varName):
            connected.update(atom.indices)

    print("open indices:", open_indices)
    print("indices connected to", varName, ":", tuple(connected))

    # Contract (sum) dummy indices that are connected to varName
    contracted = expr
    for idx in connected:
        if counts.get(idx, 0) > 1 and hasattr(idx, "upper") and idx.upper is not None:
            contracted = summation(contracted, (idx, 0, idx.upper - 1))

    return contracted
from sympy import symbols, IndexedBase, Idx, Matrix,simplify
from sympy import KroneckerDelta

A = IndexedBase('A', shape=(dim1, dim1))
U = IndexedBase('U', shape=(dim1,dim1,dim1,dim1))
Us = IndexedBase('U^*', shape=(dim1,dim1,dim1,dim1))
delta = lambda i, j: KroneckerDelta(i, j)
a1,b1,d1,g1 = symbols('alpha_1 beta_1 delta_1 gamma_1', cls=Idx)
a1p,b1p,d1p,g1p = symbols('alpha_1\' beta_1\' delta_1\' gamma_1\'', cls=Idx)
expr = U[a1,b1,0,0] * U[d1,g1,0,0] * Us[a1p,b1p,0,0] * Us[d1p,g1p,0,0] * delta(a1,d1p) * delta(a1p,d1) * delta(b1,b1p) *delta(g1,g1p)
expr = expr.replace(
    lambda e: e.has(KroneckerDelta),
    lambda e: e.doit()
)

In [32]:
expr

KroneckerDelta(alpha_1, delta_1')*KroneckerDelta(alpha_1', delta_1)*KroneckerDelta(beta_1, beta_1')*KroneckerDelta(gamma_1, gamma_1')*U[alpha_1, beta_1, 0, 0]*U[delta_1, gamma_1, 0, 0]*U^*[alpha_1', beta_1', 0, 0]*U^*[delta_1', gamma_1', 0, 0]